In [1]:
# Check if autoreload is loaded and load/reload accordingly
try:
    %reload_ext autoreload
except:
    %load_ext autoreload
%autoreload 2
import numpy as np
import h5py
import yaml
import pandas as pd
from scipy.optimize import minimize

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from theory_helper_funcs import (activity_func_generator, min_mse_loss, min_mse_loss_reg, extend_sequences)


In [51]:
def load_kinetic_data(filepath: str, template_seq: str = "AAAAAAA", mutant_seq: str = "BBBBBBB"):
    """Load and preprocess kinetic model data from HDF5 file.
    
    Parameters
    ----------
    filepath : str
        Path to HDF5 file containing kinetic model simulation data
    template_seq : str, optional
        Template sequence to extract rates for (default: "AAAAAAA")
    mutant_seq : str, optional
        Mutant sequence to extract rates for (default: "BBBBBBB")
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'x': np.ndarray - One-hot encoded sequences, shape (n_samples, seq_length, 2)
        - 'y': np.ndarray - Raw activity values, shape (n_samples,)
        - 'ka_mat': np.ndarray - King-Altman pattern matrix
        - 'ka_contrib': np.ndarray - Rate contribution matrix
        - 'target_rates': list - Rates for template sequence
        - 'mut_rates': list - Rates for mutant sequence
        - 'sequences': np.ndarray - Original sequences as strings
    
    Examples
    --------
    >>> data = load_kinetic_data("ring_datasets/test_4ring_random_0.h5")
    >>> x_train, y_train = data['x'], data['y']
    >>> print(f"Loaded {len(x_train)} sequences of length {x_train.shape[1]}")
    """
    with h5py.File(filepath, "r") as f:
        # Load sequence and activity data
        sequences = f["data"]['seq'][:].astype(str)
        y = f["data"]['raw_activity'][:]
        
        # Load King-Altman matrices
        ka_mat = f["ka_pattern_matrix"][:]
        ka_contrib = f["rate_contrib_matrix"][:]
        
        # Find target and mutant rows
        target_row = np.where(sequences == template_seq)[0][0]
        mut_row = np.where(sequences == mutant_seq)[0][0]
        
        # Extract rates (columns 1 through -2, excluding seq and activities)
        target_rates = list(f["data"][target_row])[1:-2]
        mut_rates = list(f["data"][mut_row])[1:-2]
    
    # One-hot encode sequences
    enc = OneHotEncoder(categories=[["A", "B"]], sparse=False)
    
    n_samples = len(sequences)
    seq_length = len(sequences[0])
    
    # Flatten sequences for encoding
    flat = np.array([list(str(seq)) for seq in sequences]).flatten()[:, None]
    encoded = enc.fit_transform(flat)
    
    # Reshape to (n_samples, seq_length, 2)
    x = encoded.reshape(n_samples, seq_length, 2)
    
    return {
        'x': x,
        'y': y,
        'ka_mat': ka_mat,
        'ka_contrib': ka_contrib,
        'target_rates': target_rates,
        'mut_rates': mut_rates,
    }

In [ ]:

with h5py.File("/Users/alamson/projects/Elektrum/theory_notebooks/ring_datasets/test_4ring_random_0.h5", "r") as f:
    x = f["data"]['seq'][:].astype(str)
    y = f["data"]['raw_activity'][:]
    ka_mat = f["ka_pattern_matrix"][:]
    ka_contrib = f["rate_contrib_matrix"][:]
    # Find row index where the first column has AAAAAAA
    target_row = np.where(x == "AAAAAAA")[0][0]
    mut_row = np.where(x == "BBBBBBB")[0][0]
    target_rates = list(f["data"][target_row])[1:-2]
    mut_rates = list(f["data"][mut_row])[1:-2]

enc = OneHotEncoder(categories=[["A", "B"]], sparse=False)

# Encode sequences as one-hot vectors
n_samples = len(x)
seq_length = len(x[0])  # Assuming all sequences have the same length
flat = np.array([list(str(seq)) for seq in x]).flatten()[:, None]
encoded = enc.fit_transform(flat)
x = encoded.reshape(n_samples, seq_length, 2)  # shape: (n_samples, seq_length, 2)

[3.8079472, 2.293003, 7.3467402, 0.16102308, 1.6445845, 1.3706979, 0.6750278]
[3.6251984, 9.402922, 4.4383225, 9.863131, 0.34094673, 3.572855, 0.59807533]


In [ ]:
# King-Altman function and regression with binary selectors
data = load_kinetic_data(
    "/Users/alamson/projects/Elektrum/theory_notebooks/ring_datasets/test_4ring_random_0.h5",
    template_seq="AAAAAAA",
    mutant_seq="BBBBBBB"
)
x = data['x']
y = data['y']
ka_mat = data['ka_mat']

f = activity_func_generator(data['ka_mat'], data['ka_contrib'])

# ---- Step 5: Optimization to recover a, b ----
n = x.shape[1]  # Number of dimensions
theta0 = np.random.uniform(-3, 3, size=2 * n)  # Initial guess

# Run optimizer
result = minimize(
    min_mse_loss, theta0, args=(f, x, y), method="L-BFGS-B", bounds=[(-4, 4)] * (2 * n)
)

# Extract results
estimated_theta = result.x
estimated_a = estimated_theta[::2]
estimated_b = estimated_theta[1::2]

# Analyze optimization quality
print("\n" + "="*50)
print("OPTIMIZATION RESULTS")
print("="*50)
print(f"Success: {result.success}")
print(f"Message: {result.message}")
print(f"Final loss: {result.fun:.6e}")
print(f"Iterations: {result.nit}")
print(f"Function evals: {result.nfev}")

if hasattr(result, 'jac'):
    grad_norm = np.linalg.norm(result.jac)
    print(f"Gradient norm: {grad_norm:.6e}")

print("True k_target:", data['target_rates'])
print("Estimated k_target:", np.exp(estimated_a))
print("True k_mut:", data['mut_rates'])
print("Estimated k_mut:", np.exp(estimated_b))



OPTIMIZATION RESULTS
Success: True
Message: b'CONVERGENCE: REL_REDUCTION_OF_F_<=_FACTR*EPSMCH'
Final loss: 5.466521e-06
Iterations: 144
Function evals: 2490
Gradient norm: 4.917458e-05
True k_target: [3.8079472, 2.293003, 7.3467402, 0.16102308, 1.6445845, 1.3706979, 0.6750278]
Estimated k_target: [5.50106134 0.95801459 5.36646157 0.16282365 1.56095605 1.20497191
 0.64101323]
True k_mut: [3.6251984, 9.402922, 4.4383225, 9.863131, 0.34094673, 3.572855, 0.59807533]
Estimated k_mut: [5.15876026 8.37578636 3.24384148 9.89336491 0.32373521 3.18945066
 0.56863903]


In [8]:
# Different model used for inference

ka_mat_kp = np.loadtxt("../test_files/test_4proofreading/test_4proofreading_ka_mat.nptxt")
ka_contrib_kp = np.loadtxt("../test_files/test_4proofreading/test_4proofreading_rate_contrib_mat.nptxt")

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=777
)

x_train_kp = extend_sequences(x_train, ka_mat_kp.shape[1]-ka_mat.shape[1])

n = x_train_kp.shape[1]  # Number of dimensions

f = activity_func_generator(ka_mat_kp, ka_contrib_kp)


theta0 = np.random.uniform(-3, 3, size=2 * n)  # Initial guess

# Run optimizer
result = minimize(
    min_mse_loss_reg, theta0, args=(f, x_train_kp, y_train, 1e-7), method="L-BFGS-B", bounds=[(-10, 10)] * (2 * n)
)

# Extract results
estimated_theta = result.x
estimated_a = estimated_theta[::2]
estimated_b = estimated_theta[1::2]

# Analyze optimization quality
print("\n" + "="*50)
print("OPTIMIZATION RESULTS")
print("="*50)
print(f"Success: {result.success}")
print(f"Message: {result.message}")
print(f"Final loss: {result.fun:.6e}")
print(f"Iterations: {result.nit}")
print(f"Function evals: {result.nfev}")

if hasattr(result, 'jac'):
    grad_norm = np.linalg.norm(result.jac)
    print(f"Gradient norm: {grad_norm:.6e}")

# print("True a:", true_a)
print(f"Estimated k_target:{np.exp(estimated_a):.4g}")
# print("True b:", true_b)
print(f"Estimated k_mut:{np.exp(estimated_b):.4g}")




OPTIMIZATION RESULTS
Success: True
Message: b'CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL'
Final loss: 6.021409e-05
Iterations: 287
Function evals: 5440
Gradient norm: 1.578023e-05


TypeError: unsupported format string passed to numpy.ndarray.__format__

## Neural Network Approach 

In [5]:
tf.keras.backend.clear_session()


# Custom layer for s -> k mapping
class BinarySelectorLayer(Layer):
    def __init__(self, units, **kwargs):
        super(BinarySelectorLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.a = self.add_weight(
            shape=(self.units,),
            initializer=tf.keras.initializers.RandomUniform(
                minval=0.1, maxval=2.0
            ),  # Positive range
            trainable=True,
            constraint=NonNeg(),
            name="a",
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer=tf.keras.initializers.RandomUniform(
                minval=0.1, maxval=2.0
            ),  # Positive range
            trainable=True,
            constraint=NonNeg(),
            name="b",
        )

    def call(self, s):
        return (1.0 - s) * self.a + s * self.b


# Custom layer for f(k) computation with fixed output shape
class FKLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FKLayer, self).__init__(**kwargs)

    def call(self, k):
        result = (k[:, 0] * k[:, 1] * k[:, 2] + k[:, 3]) / tf.reduce_sum(k, axis=1)
        return tf.expand_dims(result, axis=-1)  # Fix output shape

    def compute_output_shape(self, input_shape):
        return (input_shape[0], 1)


# Parameters
n = 5
num_samples = 300

# Generate binary s and simulate y = f(k(s))
np.random.seed(41)
s_data = np.random.randint(0, 2, size=(num_samples, n)).astype(np.float32)

true_a = np.random.uniform(1, 2, size=n)
true_b = np.random.uniform(2, 3, size=n)


def construct_k_np(s, a, b):
    return (1 - s) * a + s * b


k_data = np.array([construct_k_np(s, true_a, true_b) for s in s_data])


# Generate y_data using numpy for simplicity
def f_k_numpy(k):
    return (k[:, 0] * k[:, 1] * k[:, 2] + k[:, 3]) / k.sum(axis=1)


y_data = f_k_numpy(k_data).reshape(-1, 1)  # Ensure proper shape

# Debug: print shapes
print("s_data shape:", s_data.shape)
print("y_data shape:", y_data.shape)

# Build Keras model
s_input = Input(shape=(n,))
k_output = BinarySelectorLayer(n)(s_input)
y_pred = FKLayer()(k_output)

model = Model(inputs=s_input, outputs=y_pred)


def mse_log_space(y_true, y_pred):
    """MSE loss in log space"""
    # epsilon = 1e-9
    epsilon = 0
    return tf.keras.losses.mean_absolute_error(
        tf.math.log(y_true + epsilon), tf.math.log(y_pred + epsilon)
    )


model.compile(optimizer="adam", loss=mse_log_space)


# Train the model
model.fit(s_data, y_data, epochs=2000, verbose=1)

# Extract learned parameters
trained_a = model.get_layer(index=1).get_weights()[0]
trained_b = model.get_layer(index=1).get_weights()[1]

# Print comparison
for i in range(n):
    print(
        f"Index {i}: True a = {true_a[i]:.3f}, Trained a = {trained_a[i]:.3f}, "
        f"True b = {true_b[i]:.3f}, Trained b = {trained_b[i]:.3f}"
    )

s_data shape: (300, 5)
y_data shape: (300, 1)

Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Epoch 1/2000
300/300 [==============================] - 0s 438us/sample - loss: 1.0480
Epoch 2/2000
300/300 [==============================] - 0s 24us/sample - loss: 1.0281
Epoch 3/2000
300/300 [==============================] - 0s 24us/sample - loss: 1.0088
Epoch 4/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.9898
Epoch 5/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.9711
Epoch 6/2000
300/300 [==============================] - 0s 26us/sample - loss: 0.9527
Epoch 7/2000
300/300 [==============================] - 0s 26us/sample - loss: 0.9350
Epoch 8/2000
300/300 [==============================] - 0s 27us/sample - loss: 0.9172
Epoch 9/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.9000
Epoch 10/2000
 32/300 [==>...........................] - ETA

2025-07-30 16:55:07.884819: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA


300/300 [==============================] - 0s 26us/sample - loss: 0.8831
Epoch 11/2000
300/300 [==============================] - 0s 26us/sample - loss: 0.8663
Epoch 12/2000
300/300 [==============================] - 0s 27us/sample - loss: 0.8499
Epoch 13/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.8337
Epoch 14/2000
300/300 [==============================] - 0s 26us/sample - loss: 0.8186
Epoch 15/2000
300/300 [==============================] - 0s 24us/sample - loss: 0.8040
Epoch 16/2000
300/300 [==============================] - 0s 26us/sample - loss: 0.7893
Epoch 17/2000
300/300 [==============================] - 0s 30us/sample - loss: 0.7751
Epoch 18/2000
300/300 [==============================] - 0s 27us/sample - loss: 0.7609
Epoch 19/2000
300/300 [==============================] - 0s 24us/sample - loss: 0.7470
Epoch 20/2000
300/300 [==============================] - 0s 27us/sample - loss: 0.7331
Epoch 21/2000
300/300 [==============================] - 

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Input
from tensorflow.keras.models import Model
from tensorflow.keras.constraints import NonNeg
import numpy as np
import tensorflow.keras.backend as K

tf.keras.backend.clear_session()


# Custom layer for s -> k mapping
class BinarySelectorLayer(Layer):
    def __init__(self, units, **kwargs):
        super(BinarySelectorLayer, self).__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        self.a = self.add_weight(
            shape=(self.units,),
            initializer=tf.keras.initializers.RandomUniform(
                minval=0.1, maxval=2.0
            ),  # Positive range
            trainable=True,
            constraint=NonNeg(),
            name="a",
        )
        self.b = self.add_weight(
            shape=(self.units,),
            initializer=tf.keras.initializers.RandomUniform(
                minval=0.1, maxval=2.0
            ),  # Positive range
            trainable=True,
            constraint=NonNeg(),
            name="b",
        )

    def call(self, s):
        return (1.0 - s) * self.a + s * self.b


# Custom layer for f(k) computation with fixed output shape
class FKLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(FKLayer, self).__init__(**kwargs)

    def call(self, k):
        result = (k[:, 0] * k[:, 1] * k[:, 2]) / tf.reduce_sum(k, axis=1)
        return tf.expand_dims(result, axis=-1)  # Fix output shape

    def compute_output_shape(self, input_shape):
        return (input_shape[0], 1)


# Parameters
n = 5
num_samples = 300

# Generate binary s and simulate y = f(k(s))
np.random.seed(44)
s_data = np.random.randint(0, 2, size=(num_samples, n)).astype(np.float32)

true_a = np.random.uniform(1, 2, size=n)
true_b = np.random.uniform(2, 3, size=n)


def construct_k_np(s, a, b):
    return (1 - s) * a + s * b


k_data = np.array([construct_k_np(s, true_a, true_b) for s in s_data])


# Generate y_data using numpy for simplicity
def f_k_numpy(k):
    return (k[:, 0] * k[:, 1] * k[:, 2]) / k.sum(axis=1)


y_data = f_k_numpy(k_data).reshape(-1, 1)  # Ensure proper shape

# Debug: print shapes
print("s_data shape:", s_data.shape)
print("y_data shape:", y_data.shape)

# Build Keras model
s_input = Input(shape=(n,))
k_output = BinarySelectorLayer(n)(s_input)
y_pred = FKLayer()(k_output)

model = Model(inputs=s_input, outputs=y_pred)
model.compile(optimizer="adam", loss="mse")

# Train the model
model.fit(s_data, y_data, epochs=2000, verbose=1)

# Extract learned parameters
trained_a = model.get_layer(index=1).get_weights()[0]
trained_b = model.get_layer(index=1).get_weights()[1]

# Print comparison
for i in range(n):
    print(
        f"Index {i}: True a = {true_a[i]:.3f}, Trained a = {trained_a[i]:.3f}, "
        f"True b = {true_b[i]:.3f}, Trained b = {trained_b[i]:.3f}"
    )

s_data shape: (300, 5)
y_data shape: (300, 1)
Epoch 1/2000
300/300 [==============================] - 0s 399us/sample - loss: 0.6415
Epoch 2/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.6372
Epoch 3/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.6328
Epoch 4/2000
300/300 [==============================] - 0s 27us/sample - loss: 0.6281
Epoch 5/2000
300/300 [==============================] - 0s 24us/sample - loss: 0.6233
Epoch 6/2000
300/300 [==============================] - 0s 27us/sample - loss: 0.6182
Epoch 7/2000
300/300 [==============================] - 0s 23us/sample - loss: 0.6130
Epoch 8/2000
300/300 [==============================] - 0s 28us/sample - loss: 0.6076
Epoch 9/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.6020
Epoch 10/2000
300/300 [==============================] - 0s 26us/sample - loss: 0.5962
Epoch 11/2000
300/300 [==============================] - 0s 25us/sample - loss: 0.5901
Epoch